# Vietnamese Student Feedback Classification

**Lecture 9 · 60 minutes**

Classify Vietnamese student feedback as negative (`0`), neutral (`1`), or positive (`2`).

Train a TF-IDF baseline, then use a pretrained language model to extract sentence embeddings. Submit your predictions to Kaggle.

## 1. Load data

Attach the competition dataset using **Add Input**.

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

files = ['train.csv', 'valid.csv', 'test.csv', 'sample_submission.csv']
roots = [Path('/kaggle/input'), Path.cwd()]

DATA_DIR = None
for root in roots:
    if not root.exists():
        continue
    for path in root.rglob('train.csv'):
        if all((path.parent / filename).exists() for filename in files):
            DATA_DIR = path.parent
            break
    if DATA_DIR is not None:
        break

if DATA_DIR is None:
    archives = [p for root in roots if root.exists()
                for p in root.rglob('kaggle_data_upload.zip')]
    if not archives:
        raise FileNotFoundError('Attach the competition dataset under Add Input.')
    DATA_DIR = Path('/kaggle/working/dataset') if Path('/kaggle/working').exists() else Path('dataset')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for name in files:
            if name in archive.namelist():
                (DATA_DIR / name).write_bytes(archive.read(name))

train = pd.read_csv(DATA_DIR / 'train.csv')
valid = pd.read_csv(DATA_DIR / 'valid.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'Train: {len(train)}, Validation: {len(valid)}, Test: {len(test)}')
display(train.head())

## 2. Explore the data

Check the label distribution and read a few examples.

In [ ]:
display(train['label'].value_counts().sort_index())
display(train[['text', 'label']].sample(min(5, len(train)), random_state=2026))

## 3. TF-IDF baseline

**TODO 1:** Train Logistic Regression and measure validation accuracy.

In [ ]:
baseline = make_pipeline(
    TfidfVectorizer(analyzer='word', ngram_range=(1, 2), max_features=20000),
    LogisticRegression(C=1.0, max_iter=1000, random_state=2026),
)

# TODO 1A: Train on train['text'] and train['label'].
# baseline.fit(...)

# TODO 1B: Evaluate on the validation set.
# valid_predictions = ...
# validation_accuracy = ...
# print('Validation accuracy:', validation_accuracy)

## 4. Pretrained sentence embeddings

**TODO 2:** Use the pretrained multilingual MiniLM encoder to obtain sentence embeddings. Train a Logistic Regression classifier and compare its validation accuracy with TF-IDF.

If the model cannot be downloaded, enable **Internet** in Kaggle Notebook settings or attach the model under **Add Input** and set `MODEL_SOURCE` to its folder.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_SOURCE = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
encoder = SentenceTransformer(MODEL_SOURCE)

# TODO 2A: Encode train, validation, and test text with encoder.encode(...).
# E_train = ...
# E_valid = ...
# E_test = ...

pretrained_model = LogisticRegression(max_iter=1000, random_state=2026)

# TODO 2B: Train pretrained_model on E_train and evaluate on E_valid.
# pretrained_model.fit(...)
# pretrained_predictions = ...
# pretrained_accuracy = ...
# print('Pretrained model validation accuracy:', pretrained_accuracy)


## 5. Submission

**TODO 3:** Compare validation accuracies, select a classifier, and predict `test.csv`. Both classifiers must be trained and evaluated.

In [ ]:
# TODO 3: Use baseline.predict(test["text"]) or pretrained_model.predict(E_test).
predictions = None

if predictions is None:
    raise ValueError('Complete TODO 3 before creating the submission.')

submission = pd.DataFrame({
    'id': test['id'],
    'label': np.asarray(predictions, dtype=int)
})

assert len(submission) == len(test)
assert submission['id'].astype(str).tolist() == sample['id'].astype(str).tolist()
assert submission['label'].isin([0, 1, 2]).all()

OUTPUT = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else Path('submission.csv')
submission.to_csv(OUTPUT, index=False)
print('Saved:', OUTPUT)
display(submission.head())

Upload `submission.csv` using **Submit Predictions** on the competition page.